## Column transfer ##

**we first did encoding and concatenated simply by using column transfer**

In [36]:
#IMPORTING BASIC LIBRARIES
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

In [37]:
#Loading dataset
df=pd.read_csv(r'D:\programming\datasets\covid_toy.csv')

df.sample(5)

,age,gender,fever,cough,city,has_covid
18,64,Female,98.0,Mild,Bangalore,Yes
94,79,Male,NaN,Strong,Kolkata,Yes
80,14,Female,99.0,Mild,Mumbai,Yes
97,20,Female,101.0,Mild,Bangalore,No
82,24,Male,98.0,Mild,Kolkata,Yes


**refer without.ipynb for data reading**

## TRAIN_TEST_SPLIT ##


In [25]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(df.drop(columns=['has_covid']) , df['has_covid'] , test_size=0.2 , random_state=47)

In [26]:
X_train.sample(3)

,age,gender,fever,cough,city
6,14,Male,101.0,Strong,Bangalore
48,66,Male,99.0,Strong,Bangalore
73,34,Male,98.0,Strong,Kolkata


## Encoding ##

**we will do one hot encoding on nomial columns** 

In [27]:
from sklearn.preprocessing import OneHotEncoder
oe=OneHotEncoder(sparse_output=False , drop='first' , dtype=np.int32)

X_train_oe=oe.fit_transform(X_train[['gender' , 'city']])
X_test_oe=oe.transform(X_test[['gender' , 'city']])

X_train_oe.shape
# 3 columns of city and 1 of gender :- as 1 from each dropped so 4 columns in total


(80, 4)

**filling out missing values in fever**

In [28]:
from sklearn.impute import SimpleImputer

#fills mean in Null values

imputer=SimpleImputer()
X_train_fever=imputer.fit_transform(X_train[['fever']])
X_test_fever=imputer.transform(X_test[['fever']])

X_train_fever


array([[ 98.        ],
       [ 99.        ],
       [101.        ],
       [100.        ],
       [ 98.        ],
       [ 98.        ],
       [ 99.        ],
       [100.        ],
       [101.        ],
       [101.        ],
       [ 98.        ],
       [101.        ],
       [104.        ],
       [100.62162162],
       [103.        ],
       [103.        ],
       [100.        ],
       [100.        ],
       [ 98.        ],
       [103.        ],
       [101.        ],
       [ 99.        ],
       [103.        ],
       [100.        ],
       [101.        ],
       [ 98.        ],
       [104.        ],
       [ 98.        ],
       [104.        ],
       [100.        ],
       [103.        ],
       [102.        ],
       [101.        ],
       [ 98.        ],
       [104.        ],
       [ 98.        ],
       [101.        ],
       [102.        ],
       [ 99.        ],
       [102.        ],
       [100.62162162],
       [100.        ],
       [103.        ],
       [100

In [29]:
X_train_fever.shape

(80, 1)

**Ordinal encoding**

In [30]:
from sklearn.preprocessing import OrdinalEncoder
ore=OrdinalEncoder(categories=[['Mild','Strong']])

X_train_cough = ore.fit_transform(X_train[['cough']])
X_test_cough = ore.transform(X_test[['cough']])

X_train_cough

array([[0.],
       [1.],
       [1.],
       [1.],
       [1.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [1.],
       [1.],
       [0.],
       [0.],
       [1.],
       [0.],
       [1.],
       [1.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [0.],
       [1.],
       [1.],
       [0.],
       [1.],
       [0.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [1.],

**Label Encoding**

In [31]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()

y_train_label=le.fit_transform(y_train)
y_test_label=le.transform(y_test)

y_train_label

array([1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0,
       1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0,
       1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1,
       1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1])

In [32]:
#conversion to dataframe
y_train_label=pd.DataFrame(y_train_label , columns=['has_covid'])
y_test_label=pd.DataFrame(y_test_label , columns=['has_covid'])

y_train_label.head()

,has_covid
0,1
1,0
2,1
3,0
4,1


## Column transfer ##

In [33]:
from sklearn.compose import ColumnTransformer

transformer = ColumnTransformer(transformers=[
    ('tnf1', SimpleImputer(), ['fever']),
    ('tnf2', OrdinalEncoder(categories=[['Mild','Strong']]), ['cough']),
    ('tnf3', OneHotEncoder(sparse_output=False, drop='first', dtype=np.int32), ['gender','city'])
], remainder='passthrough')  # untouched columns (age) are passed through

X_train_transformed = transformer.fit_transform(X_train)
X_test_transformed = transformer.transform(X_test)   

X_train_transformed.shape   

(80, 7)

In [35]:
X_test_transformed.shape

(20, 7)